In [1]:
# importing necessary packages 
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from gensim.corpora import Dictionary
from gensim.models import word2vec
from sklearn.manifold import TSNE as tsne

In [3]:
TOKENS = pd.read_csv('/Users/aniyahmcwilliams/Spring 2026/Text As Data/final project/DATA/CORPUS.csv', index_col=['mag_id','page_num','token_num'])
TOKENS

pos pos_group token_str  term_str
mag_id page_num token_num                                   
0      1        0           DT        DT       The       the
                1           NN        NN      text      text
                2           IN        IN        on        on
                3           DT        DT      this      this
                4           NN        NN      page      page
...                        ...       ...       ...       ...
9      68       55         NNP        NN      INC.       inc
                57         NNP        NN  Michigan  michigan
                58         NNP        NN    Avenue    avenue
                60         NNP        NN   Chicago   chicago
                62         NNP        NN  Illinois  illinois

[131948 rows x 4 columns]

In [4]:
# need to exclude proper nouns from the table 
TOKENS = TOKENS[TOKENS['pos'] != 'NNP']
TOKENS = TOKENS[TOKENS['pos'] != 'NNPS']
TOKENS

pos pos_group   token_str    term_str
mag_id page_num token_num                                      
0      1        0          DT        DT         The         the
                1          NN        NN        text        text
                2          IN        IN          on          on
                3          DT        DT        this        this
                4          NN        NN        page        page
...                        ..       ...         ...         ...
9      68       43         NN        NN  popularity  popularity
                45         CC        CC       write       write
                46         NN        NN       today       today
                47         IN        IN         for         for
                48         JJ        JJ     Special     special

[99929 rows x 4 columns]

## Converting to Genism 

In [ ]:
docs = TOKENS.groupby('mag_id').term_str.apply(list).tolist()
# for i in range(5):
#     print(f"Magazine {i}:", docs[i])

Magazine 0: ['the', 'text', 'on', 'this', 'page', 'is', 'estimated', 'to', 'be', 'only', 'accurate', 'nrwt', 'ii', 'kl', 'll', 'll', 'warning', 'the', 'determined', 'that', 'is', 'dangerous', 'to', 'your', 'and', 'jmgsr', 'jvm', 'editor', 'and', 'photo', 'by', 'and', 'of', 'story', 'on', 'page', 'african', 'and', 'and', 'people', 'people', 'are', 'talking', 'about', 'politics', 'readers', 'of', 'the', 'weekly', 'by', 'office', 'at', 'of', 'the', 'office', 'secondclass', 'postage', 'paid', 'at', 'by', 'one', 'year', 'foreign', 'we', 'can', 'not', 'be', 'responsible', 'for', 'unsolicited', 'material', 'of', 'readers', 'discussed', 'regarding', 'your', 'story', 'on', 'recalled', 'the', 'many', 'articles', 'which', 'have', 'read', 'on', 'the', 'super', 'star', 'and', 'none', 'really', 'tell', 'where', 'he', 'lived', 'and', 'how', 'his', 'first', 'single', 'became', 'mild', 'hit', 'maybe', 'green', 'doesn', 'want', 'anyone', 'to', 'know', 'that', 'he', 'is', 'from', 'and', 'his', 'family', 

## Extract vocabulary

In [9]:
dictionary = Dictionary(docs)
dictionary

## Generating Word Embeddings

In [21]:
w2v_params = dict(
    window = 2,
    vector_size = 100,
    min_count = 25, # THIS LIMITS OUR VOCAB
    workers = 4
)

In [22]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

array([[-0.11116361,  0.14879623,  0.07234195, ..., -0.11253504,
        -0.05682641,  0.00954275],
       [-0.13956845,  0.16889782,  0.05354016, ..., -0.12051115,
        -0.07743851,  0.00326741],
       [-0.10400221,  0.09931282,  0.02901649, ..., -0.09494261,
        -0.09555712, -0.01172785],
       ...,
       [-0.0769724 ,  0.06115159,  0.02838993, ..., -0.06099846,
        -0.09873429, -0.00514937],
       [-0.05572275,  0.05826187,  0.01322086, ..., -0.06619602,
        -0.08180808, -0.0143179 ],
       [-0.07894122,  0.05390297,  0.02000701, ..., -0.07209023,
        -0.09371654, -0.00381032]], dtype=float32)

In [23]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.111164,0.148796,0.072342,0.221217,-0.096961,-0.453041,0.112179,0.441204,-0.259200,-0.223016,...,0.211859,0.048012,0.042461,0.031390,0.349741,0.213687,0.117057,-0.112535,-0.056826,0.009543
of,-0.139568,0.168898,0.053540,0.209348,-0.061916,-0.469744,0.121293,0.460138,-0.280049,-0.243090,...,0.203799,0.035587,0.040625,0.009337,0.313355,0.193293,0.175569,-0.120511,-0.077439,0.003267
and,-0.104002,0.099313,0.029016,0.169570,-0.068836,-0.371445,0.085844,0.390691,-0.233965,-0.143484,...,0.246745,0.051926,0.055386,0.052918,0.338773,0.231648,0.122375,-0.094943,-0.095557,-0.011728
in,-0.121550,0.129748,0.043063,0.183863,-0.066226,-0.413353,0.108203,0.415633,-0.230196,-0.194259,...,0.246074,0.043586,0.040080,0.038126,0.336433,0.209372,0.142846,-0.112944,-0.098142,-0.007254
to,-0.096134,0.081485,0.018845,0.170819,-0.053158,-0.333459,0.099150,0.370564,-0.217655,-0.101219,...,0.310803,0.050346,0.052373,0.059524,0.324704,0.223469,0.118506,-0.072448,-0.141291,-0.000528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
college,-0.070985,0.055259,0.026298,0.119840,-0.036666,-0.244972,0.063084,0.266076,-0.160929,-0.083292,...,0.198085,0.038669,0.022220,0.049246,0.218650,0.156782,0.081107,-0.061799,-0.093526,0.000318
campus,-0.075425,0.061981,0.026304,0.126004,-0.054484,-0.264057,0.068934,0.260995,-0.156575,-0.085737,...,0.187879,0.027633,0.039328,0.045370,0.222851,0.158528,0.084392,-0.059340,-0.083213,-0.007203
born,-0.076972,0.061152,0.028390,0.134752,-0.054440,-0.256888,0.074180,0.274449,-0.150559,-0.088570,...,0.212473,0.030916,0.022877,0.056512,0.219704,0.162631,0.076506,-0.060998,-0.098734,-0.005149


# tSNE Visualization

In [27]:
PP = 15

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='pca', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)
TSNE

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning:

divide by zero encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning:

overflow encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning:

invalid value encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning:

divide by zero encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning:

overflow encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/lib/python3.12/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning:

invalid value encountered in matmul

/Users/aniyahmcwilliams/miniconda3/envs/ds6021/l

,x,y
term_str,,
the,-22.972446,30.528456
of,-24.091766,30.405582
and,-28.738213,25.330706
in,-24.114258,27.399509
to,-52.326359,-6.484911
...,...,...
college,90.373253,-3.352135
campus,92.160866,0.820726
born,81.963188,-1.952649


In [28]:
TSNE['gid'] = TSNE.apply(lambda x: dictionary.token2id[x.name], axis=1)
TSNE['n'] = TSNE.gid.map(dictionary.cfs)
TSNE

,x,y,gid,n
term_str,,,,
the,-22.972446,30.528456,2623,6877
of,-24.091766,30.405582,1750,3809
and,-28.738213,25.330706,115,3186
in,-24.114258,27.399509,1309,2992
to,-52.326359,-6.484911,2664,2978
...,...,...,...,...
college,90.373253,-3.352135,507,25
campus,92.160866,0.820726,382,25
born,81.963188,-1.952649,319,25


In [29]:
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        # size='n',
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')